In [26]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_API_KEY')

In [6]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv langchain-google-genai langchain langchain-text-splitters



In [12]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [20]:
video_id="wfJpKjMwXpg"
try:
  yt_api = YouTubeTranscriptApi()


  translist = yt_api.fetch(video_id)

  transcript=" ".join(chunk.text for chunk in translist)
  print(transcript)
except TranscriptsDisabled:
  print("Transcript disabled")

It's my honor to come over
here to talk a little bit about embodied intelligence. It's something my
lab has been thinking of over the past few months-ish. We generally work
with morphing matter. We recently started to
think about the possibility of leveraging morphing
materials and mechanism, basically to think about
embodied intelligence from our perspective. So embodied intelligence,
I'm sure you guys have heard of it under different contexts. Perhaps it means many
different things. It's vaguely defined
at this moment. So from the context
of this talk, we are pretty much talking about
some level of programmability or decision-making
through hardware system, especially in our case
through shape changing and tunable materials
and structures. So people argue
those systems are useful from different
perspectives. You could argue this kind of
purely mechanical systems that can do computation or
some sort of actuation is good for physical
cybersecurity. So the idea is there are
certain impo

In [21]:
splitt=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitt.create_documents([transcript])

In [22]:
len(chunks)

65

In [23]:
chunks[60]

Document(metadata={}, page_content="system basically rolling around and collect DNA as\nlong as we know roughly which area this robot covers. [? It's ?] where\nwe are thinking. But no exact project at\nthis moment is going on. Unless you have some good ideas. [CHUCKLES] Yeah, and also this image is\nshowing the flying machine. So this is something\nactually one of my students is working on, basically like\ndandelion and milkweed-inspired flyers, also passive. So those, [? you ?]\ncould potentially use it to collect environmental\nDNAs in the midair versus the hoppers. And the rollers will be\non the ground, basically. [INAUDIBLE] question,\nso your work has done so many different\nsource for actuation, such as kinetic and\na chemical reaction [INAUDIBLE] temperature. I'm curious, do you\nhave a favorite, or what are some other\npotential modality of actuation that you will be excited\nto explore in the future? At this point, I feel\nlike motor is awesome. [CHUCKLES] And in terms")

In [27]:
embedding=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
vector_Store=FAISS.from_documents(chunks,embedding)

In [28]:
vector_Store.index_to_docstore_id

{0: '3ca0f8d1-b837-4335-9362-a6a68ede8e7a',
 1: '9f9fbfab-4fe3-448b-9179-a777a81e8e7c',
 2: '1ec7ba1d-c6b1-4130-b02b-da36e6351267',
 3: '77f6b713-5c91-4b1f-bb39-bf2ef7d0d2f4',
 4: 'a17d2ffb-103f-4c81-a15c-aeae3ed7de3a',
 5: '91e071ac-6455-49a4-83c4-2925b3092c9d',
 6: 'c87d9cb7-3a51-44ac-a891-2bd08774eecc',
 7: '40890130-a679-487f-b9f3-582c84c0f835',
 8: '2e62aa92-db53-483c-b92d-9ec29c337590',
 9: '1c7158e3-d470-43b3-9f02-fa3e4c9e67e5',
 10: 'eeaa03b0-1507-4aa5-9d76-c7644d1fe5f6',
 11: '99e3099a-16bd-4866-9ea4-e7cea838ea6f',
 12: 'e47d92f3-024f-4769-bc27-47a48c7220a3',
 13: '4cc47ad0-466e-4e03-a77c-bc9bca62c21b',
 14: 'e59448a2-7259-4fb2-94c5-455ecb59a0d5',
 15: '123d4843-8ded-4a47-9863-8512074f51f5',
 16: 'e19d1294-0e0a-490e-8d72-1550632cf9f8',
 17: '6d69a785-353f-4625-8881-fb8ebc67ab45',
 18: '242938c7-c4bc-4424-89d4-adb20ca16c4a',
 19: '9da61e37-6761-4f2f-8b49-0b6a76240f46',
 20: '5f4ab442-b605-4b69-8f2b-866c321376a0',
 21: 'bd988b4a-3e37-4e39-9edd-9bd84177bf06',
 22: '14578dcf-6f03-

In [32]:
first_doc_id = list(vector_Store.docstore._dict.keys())[0]
first_document = vector_Store.docstore._dict[first_doc_id]

print("Text Metadata inside Vector Store:")
print(first_document.page_content)


Text Metadata inside Vector Store:
It's my honor to come over
here to talk a little bit about embodied intelligence. It's something my
lab has been thinking of over the past few months-ish. We generally work
with morphing matter. We recently started to
think about the possibility of leveraging morphing
materials and mechanism, basically to think about
embodied intelligence from our perspective. So embodied intelligence,
I'm sure you guys have heard of it under different contexts. Perhaps it means many
different things. It's vaguely defined
at this moment. So from the context
of this talk, we are pretty much talking about
some level of programmability or decision-making
through hardware system, especially in our case
through shape changing and tunable materials
and structures. So people argue
those systems are useful from different
perspectives. You could argue this kind of
purely mechanical systems that can do computation or
some sort of actuation is good for physical
cybersecurity. So

In [33]:
retriever=vector_Store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [36]:
llm=ChatGoogleGenerativeAI(model='gemini-3.6-flash', temperature=0.2)

In [38]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [57]:
question='if the topic of ai discussed in this video? if yes what was discussed '
retreivedocs=retriever.invoke(question)

In [58]:
retreivedocs

[Document(id='3d93c30f-9800-4519-adf8-37ed9ae09a90', metadata={}, page_content="arm driven by motors per se, with a very\nclassic robust design. And ultimately, those\nmachines give us the most precise control,\nthe amount of torque that's more ideal for\nheavy-duty things. And it's fully\ncontrollable in a sense. You can easily plug\nin machine learning or sophisticated control\npolicy to control those robot. And mechanical systems or\nmaterial-driven systems are the opposite. They are slow. They're not precise. They're very weak. Absolutely in the field of soft\nrobots and basically mechanical intelligence, we discuss a lot\nwhether [? they ?] [? are ?] killer app at all for\nmechanical systems. I think we more or less started\nto reach the conclusion maybe there is something\nthat can combine both mechanical intelligence\nand computational intelligence and reach somewhere. Maybe neither could\nachieve by itself. This is super experimental,\nexploratory thoughts. I actually can't rea

In [59]:
context_text="\n\n".join(doc.page_content for doc in retreivedocs)
final_prompt=prompt.invoke({'context':context_text,'question':question})

In [60]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      arm driven by motors per se, with a very\nclassic robust design. And ultimately, those\nmachines give us the most precise control,\nthe amount of torque that's more ideal for\nheavy-duty things. And it's fully\ncontrollable in a sense. You can easily plug\nin machine learning or sophisticated control\npolicy to control those robot. And mechanical systems or\nmaterial-driven systems are the opposite. They are slow. They're not precise. They're very weak. Absolutely in the field of soft\nrobots and basically mechanical intelligence, we discuss a lot\nwhether [? they ?] [? are ?] killer app at all for\nmechanical systems. I think we more or less started\nto reach the conclusion maybe there is something\nthat can combine both mechanical intelligence\nand computational intelligence and reach somewhere. 

In [61]:
answer=llm.invoke(final_prompt)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [62]:
answer.content[0]['text']

'Yes, the topic of AI (and related concepts like machine learning and embodied intelligence) is discussed in the transcript:\n\n* **Machine Learning & Computational Intelligence:** The speaker mentions that you can easily plug machine learning or a sophisticated control policy into motor-driven robotic machines to control them. They also discuss combining mechanical intelligence with computational intelligence to achieve results that neither could achieve alone.\n* **Ecological Physical AI:** The speaker introduces "ecological physical AI" near the end, describing it in the context of purely ambient energy-powered field robots. These systems would be made cheaply without electronics, deployed massively, require minimum intelligence to execute tasks, and eventually degrade into the environment.\n* **Embodied and Hybrid Intelligence:** The transcript discusses "hybrid intelligence" (using smart materials to augment precise, controllable machines) and "embodied intelligence" (which can be

In [63]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [65]:
def formaldocs(retreivedocs):
  context_text="\n\n".join(doc.page_content for doc in retreivedocs)
  return context_text

In [66]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(formaldocs),
    'question': RunnablePassthrough()
})

In [67]:
parallel_chain.invoke("who is ai")

{'context': "arm driven by motors per se, with a very\nclassic robust design. And ultimately, those\nmachines give us the most precise control,\nthe amount of torque that's more ideal for\nheavy-duty things. And it's fully\ncontrollable in a sense. You can easily plug\nin machine learning or sophisticated control\npolicy to control those robot. And mechanical systems or\nmaterial-driven systems are the opposite. They are slow. They're not precise. They're very weak. Absolutely in the field of soft\nrobots and basically mechanical intelligence, we discuss a lot\nwhether [? they ?] [? are ?] killer app at all for\nmechanical systems. I think we more or less started\nto reach the conclusion maybe there is something\nthat can combine both mechanical intelligence\nand computational intelligence and reach somewhere. Maybe neither could\nachieve by itself. This is super experimental,\nexploratory thoughts. I actually can't really\nthink about very convincing robotic system that shows\n\nIt's 

In [68]:
parser=StrOutputParser()

In [69]:
main_chain=parallel_chain| prompt|llm|parser

In [70]:
main_chain.invoke("can you summarize the video")

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the transcript context, several video demonstrations are mentioned:\n\n1. **Thermally responsive pump and moisture valve:** A video demonstrating a component that harvests thermal fluctuations in the air to accumulate compressed air into a chamber, working alongside a moisture-sensitive "kink valve" designed for an automated garden that operates without electricity.\n2. **Erodium seed mechanism:** A video illustrating how a moisture-responsive coiled body (inspired by the *erodium* seed) unwinds in the rain. This unwinding produces rotational motion and downward thrust to push a seed tip into the soil.\n3. **Grooved material morphing:** A video showing a side-by-side comparison of an experiment and a simulation. It demonstrates a flat piece of material placed in water that bends toward its grooved side, reaches equilibrium, and then bends in the opposite direction when taken out of the water.'